In [ ]:
import pandas as pd
from collections import defaultdict

train = pd.read_csv('data/train.csv')
train.head()

In [ ]:
# key: output 단어, value: 같은 위치의 input 단어 집합
word_dict = defaultdict(set)
mismatch_ids = []  # 스플릿 후 단어 수가 다른 행

for id_, inp, out in zip(train['ID'], train['input'], train['output']):
    # output 끝에 공백이 붙은 행이 있어 split(' ') 대신 split() 사용
    inp_words = inp.split()
    out_words = out.split()
    if len(inp_words) != len(out_words):
        mismatch_ids.append(id_)
        continue
    for inp_word, out_word in zip(inp_words, out_words):
        word_dict[out_word].add(inp_word)

word_dict = dict(word_dict)

print('단어 수(key):', len(word_dict))
print('단어 수가 달라 제외된 행:', len(mismatch_ids))

In [ ]:
word_dict.keys()

In [ ]:
import json

# 한 줄에 단어 하나씩 저장 (set은 JSON으로 저장할 수 없어 정렬된 리스트로 변환)
with open('data/word_dict.jsonl', 'w', encoding='utf-8') as f:
    for out_word, inp_words in word_dict.items():
        f.write(json.dumps({'output': out_word, 'input': sorted(inp_words)}, ensure_ascii=False) + '\n')

In [ ]:
import json

CHOSUNG = 'ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎ'
# 된소리/거센소리는 예사소리와 같은 초성으로 취급
SAME_CHOSUNG = {'ㄲ': 'ㄱ', 'ㅋ': 'ㄱ', 'ㄸ': 'ㄷ', 'ㅌ': 'ㄷ', 'ㅃ': 'ㅂ', 'ㅍ': 'ㅂ', 'ㅆ': 'ㅅ', 'ㅉ': 'ㅈ', 'ㅊ': 'ㅈ'}

def get_chosung(word):
    # 한글 음절은 초성으로, 그 외 문자(숫자, 문장부호 등)는 그대로 둠
    result = []
    for ch in word:
        if '가' <= ch <= '힣':
            cho = CHOSUNG[(ord(ch) - ord('가')) // 588]
            result.append(SAME_CHOSUNG.get(cho, cho))
        else:
            result.append(ch)
    return ''.join(result)

n_words = 0
n_diff_words = 0
with open('data/word_dict.jsonl', encoding='utf-8') as f_in, \
     open('data/chosung_diff.jsonl', 'w', encoding='utf-8') as f_out:
    for line in f_in:
        row = json.loads(line)
        n_words += 1
        out_cho = get_chosung(row['output'])
        # 초성이 하나라도 다른 input 단어만 모음
        diff_words = [w for w in row['input'] if get_chosung(w) != out_cho]
        if diff_words:
            n_diff_words += 1
            f_out.write(json.dumps({'output': row['output'], 'input': diff_words}, ensure_ascii=False) + '\n')

print('전체 단어 수:', n_words)
print('초성이 다른 input이 있는 단어 수:', n_diff_words)

In [ ]:
MIN_DIFF = 2  # 초성이 이 개수 이상 다른 경우만 저장

def count_chosung_diff(word_a, word_b):
    # 같은 위치의 초성을 비교해 다른 글자 수를 셈 (길이가 다르면 남는 글자도 다른 것으로 셈)
    cho_a, cho_b = get_chosung(word_a), get_chosung(word_b)
    n_diff = sum(a != b for a, b in zip(cho_a, cho_b))
    return n_diff + abs(len(cho_a) - len(cho_b))

n_words = 0
n_diff_words = 0
with open('data/word_dict.jsonl', encoding='utf-8') as f_in, \
     open('data/chosung_diff_2.jsonl', 'w', encoding='utf-8') as f_out:
    for line in f_in:
        row = json.loads(line)
        n_words += 1
        diff_words = [w for w in row['input'] if count_chosung_diff(row['output'], w) >= MIN_DIFF]
        if diff_words:
            n_diff_words += 1
            f_out.write(json.dumps({'output': row['output'], 'input': diff_words}, ensure_ascii=False) + '\n')

print('전체 단어 수:', n_words)
print(f'초성이 {MIN_DIFF}글자 이상 다른 input이 있는 단어 수:', n_diff_words)

In [ ]:
def has_jongsung(ch):
    return '가' <= ch <= '힣' and (ord(ch) - ord('가')) % 28 != 0

def is_yeoneum_pos(word, i):
    # 연음 위치: 앞 글자에 받침이 있고 현재 글자의 초성이 ㅇ인 경우 (예: 사람들'이', 겪'어'본)
    return i > 0 and get_chosung(word[i]) == 'ㅇ' and has_jongsung(word[i - 1])

def has_non_yeoneum_diff(out_word, inp_word):
    # 연음 위치가 아닌 곳에서 초성이 바뀌었는지 검사 (길이가 다르면 위치 비교가 안 되므로 바뀐 것으로 봄)
    if len(out_word) != len(inp_word):
        return True
    return any(
        get_chosung(o) != get_chosung(i) and not is_yeoneum_pos(out_word, pos)
        for pos, (o, i) in enumerate(zip(out_word, inp_word))
    )

n_words = 0
n_diff_words = 0
with open('data/word_dict.jsonl', encoding='utf-8') as f_in, \
     open('data/chosung_diff_not_yeoneum.jsonl', 'w', encoding='utf-8') as f_out:
    for line in f_in:
        row = json.loads(line)
        n_words += 1
        diff_words = [w for w in row['input'] if has_non_yeoneum_diff(row['output'], w)]
        if diff_words:
            n_diff_words += 1
            f_out.write(json.dumps({'output': row['output'], 'input': diff_words}, ensure_ascii=False) + '\n')

print('전체 단어 수:', n_words)
print('연음 위치가 아닌 곳에서 초성이 바뀐 input이 있는 단어 수:', n_diff_words)

In [ ]:
def is_hangul(ch):
    return '가' <= ch <= '힣'

def has_diff_after_no_jongsung(out_word, inp_word):
    # 받침이 없는 글자 바로 뒤 글자의 초성이 바뀌었는지 검사 (예: '너무'의 '무')
    if len(out_word) != len(inp_word):
        return False
    return any(
        is_hangul(out_word[pos - 1]) and not has_jongsung(out_word[pos - 1])
        and get_chosung(out_word[pos]) != get_chosung(inp_word[pos])
        for pos in range(1, len(out_word))
    )

n_words = 0
n_diff_words = 0
with open('data/word_dict.jsonl', encoding='utf-8') as f_in, \
     open('data/chosung_diff_no_jongsung.jsonl', 'w', encoding='utf-8') as f_out:
    for line in f_in:
        row = json.loads(line)
        n_words += 1
        diff_words = [w for w in row['input'] if has_diff_after_no_jongsung(row['output'], w)]
        if diff_words:
            n_diff_words += 1
            f_out.write(json.dumps({'output': row['output'], 'input': diff_words}, ensure_ascii=False) + '\n')

print('전체 단어 수:', n_words)
print('받침 없는 글자 뒤에서 초성이 바뀐 input이 있는 단어 수:', n_diff_words)

In [ ]:
JUNGSUNG = 'ㅏㅐㅑㅒㅓㅔㅕㅖㅗㅘㅙㅚㅛㅜㅝㅞㅟㅠㅡㅢㅣ'
JONGSUNG = ['', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']

def get_jungsung(ch):
    return JUNGSUNG[(ord(ch) - ord('가')) % 588 // 28] if is_hangul(ch) else ''

def get_jongsung(ch):
    return JONGSUNG[(ord(ch) - ord('가')) % 28] if is_hangul(ch) else ''

def is_gugaeum_pos(word, i):
    # 구개음화 위치: 앞 글자 받침이 ㄷ/ㅌ(ㄾ)이고 현재 글자가 이/여/히/혀 등인 경우 (예: 같'이', 붙'여', 닫'히'다)
    return (
        i > 0
        and get_jongsung(word[i - 1]) in ('ㄷ', 'ㅌ', 'ㄾ')
        and get_chosung(word[i]) in ('ㅇ', 'ㅎ')
        and get_jungsung(word[i]) in ('ㅣ', 'ㅑ', 'ㅕ', 'ㅛ', 'ㅠ')
    )

def has_gugaeum(out_word, inp_word):
    # 구개음화 위치에서 input 초성이 ㅈ/ㅉ/ㅊ으로 바뀌었는지 검사 (예: 같이 -> 가치)
    if len(out_word) != len(inp_word):
        return False
    return any(
        is_gugaeum_pos(out_word, pos) and get_chosung(inp_word[pos]) == 'ㅈ'
        for pos in range(1, len(out_word))
    )

n_words = 0
n_diff_words = 0
with open('data/word_dict.jsonl', encoding='utf-8') as f_in, \
     open('data/chosung_diff_gugaeum.jsonl', 'w', encoding='utf-8') as f_out:
    for line in f_in:
        row = json.loads(line)
        n_words += 1
        diff_words = [w for w in row['input'] if has_gugaeum(row['output'], w)]
        if diff_words:
            n_diff_words += 1
            f_out.write(json.dumps({'output': row['output'], 'input': diff_words}, ensure_ascii=False) + '\n')

print('전체 단어 수:', n_words)
print('구개음화가 일어난 input이 있는 단어 수:', n_diff_words)

In [ ]:
JUNGSUNG = 'ㅏㅐㅑㅒㅓㅔㅕㅖㅗㅘㅙㅚㅛㅜㅝㅞㅟㅠㅡㅢㅣ'
JONGSUNG = ['', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ']

def get_jungsung(ch):
    return JUNGSUNG[(ord(ch) - ord('가')) % 588 // 28] if is_hangul(ch) else ''

def get_jongsung(ch):
    return JONGSUNG[(ord(ch) - ord('가')) % 28] if is_hangul(ch) else ''

def is_gugaeum_pos(word, i):
    # 구개음화 위치: 앞 글자 받침이 ㄷ/ㅌ(ㄾ)이고 현재 글자가 이/여/히/혀 등인 경우 (예: 같'이', 붙'여', 닫'히'다)
    return (
        i > 0
        and get_jongsung(word[i - 1]) in ('ㄷ', 'ㅌ', 'ㄾ')
        and get_chosung(word[i]) in ('ㅇ', 'ㅎ')
        and get_jungsung(word[i]) in ('ㅣ', 'ㅑ', 'ㅕ', 'ㅛ', 'ㅠ')
    )

def has_gugaeum(out_word, inp_word):
    # 구개음화 위치에서 input 초성이 ㅈ/ㅉ/ㅊ으로 바뀌었는지 검사 (예: 같이 -> 가치)
    if len(out_word) != len(inp_word):
        return False
    return any(
        is_gugaeum_pos(out_word, pos) and get_chosung(inp_word[pos]) == 'ㅈ'
        for pos in range(1, len(out_word))
    )

In [ ]:
n_words = 0
n_diff_words = 0
with open('data/word_dict.jsonl', encoding='utf-8') as f_in, \
     open('data/chosung_diff_gugaeum.jsonl', 'w', encoding='utf-8') as f_out:
    for line in f_in:
        row = json.loads(line)
        n_words += 1
        diff_words = [w for w in row['input'] if has_gugaeum(row['output'], w)]
        if diff_words:
            n_diff_words += 1
            f_out.write(json.dumps({'output': row['output'], 'input': diff_words}, ensure_ascii=False) + '\n')

print('전체 단어 수:', n_words)
print('구개음화가 일어난 input이 있는 단어 수:', n_diff_words)

In [ ]:
word_dict['사장님']

In [ ]:
# 변형 종류가 많은 단어 상위 20개
sorted(word_dict.items(), key=lambda x: len(x[1]), reverse=True)[:20]